# PySpark Classification Project (Big Data)

**Dataset:** Chicago_Crimes.csv (≥1GB)

This notebook implements a reproducible big-data pipeline in **PySpark**: ingestion → validation → Parquet lakehouse (Bronze/Silver/Gold) → EDA → feature engineering (custom Transformer) → distributed model training and tuning (≥4 classifiers) → evaluation and interpretability → exports for Tableau and reporting.

Notes:
- The main pipeline and models run in PySpark.
- A **single-node scikit-learn baseline** is included on a sampled dataset for comparison and serialization requirements.


In [1]:
# Dependencies
!pip -q install --no-deps mlflow databricks-sdk gdown


In [2]:
# Section: Imports
import os, re, json, time, math, random, glob, csv
from datetime import datetime, timezone
import numpy as np
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, functions as F, types as T, Row
from pyspark.sql.window import Window
from pyspark.storagelevel import StorageLevel

from pyspark.ml import Pipeline, Transformer
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier, DecisionTreeClassifier, LinearSVC
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

from pyspark.ml.param.shared import Param, Params, TypeConverters
from pyspark.ml.util import DefaultParamsReadable, DefaultParamsWritable
from pyspark.ml.linalg import VectorUDT

import mlflow
import mlflow.spark
import pickle

from pyspark.sql.functions import udf

@udf(returnType=T.ArrayType(T.DoubleType()))
def vector_to_list(v):
    if v is None:
        return None
    return v.toArray().tolist()


In [3]:
# Section: CONFIG
CONFIG = {
    # Data acquisition
    "drive_mount": True,
    "raw_csv_path": "/content/drive/MyDrive/Chicago_Crimes.csv",   # <-- update if needed
    "gdrive_file_id": "1v906i352jseHeBoLDfwUqwZ-W98vhYHI",        # optional gdown fallback
    "download_if_missing": False,

    # Output roots
    "project_root": "/content/block3_chicago_crimes_pyspark",
    "event_log_dir": "/content/block3_chicago_crimes_pyspark/spark-events",
    "checkpoint_dir": "/content/block3_chicago_crimes_pyspark/checkpoints",

    # Spark performance defaults
    "shuffle_partitions": 200,
    "use_gpu": 0,  # ETL only; requires RAPIDS Spark SQL plugin installed

    # Target
    "target_col": "is_arrest",
    "seed": 42,

    # Correlation/heatmap sampling
    "corr_sample_fraction": 0.02,

    # Tuning compute constraints
    "cv_folds": 2,
    "cv_parallelism": 2,
    "tuning_target_rows": 80000,

    # Temporal split quantiles
    "split_quantiles": (0.90, 0.95),

    # Evaluation + significance
    "bootstrap_n": 200,
    "bootstrap_sample_max_rows": 200000,

    # Business cost alignment
    "fp_cost": 1.0,
    "fn_cost": 5.0,

    # Explainability
    "lime_num_perturb": 2000,
    "lime_sigma_frac": 0.15,
    "lime_kernel_width": 0.75,
    "shap_samples": 200,
    "shap_mc_per_row": 6,
}

RUN_ID = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
print('RUN_ID:', RUN_ID)

RUN_ID: 20260304_152413


/tmp/ipykernel_155039/712967200.py:49: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_ID = datetime.utcnow().strftime('%Y%m%d_%H%M%S')


In [6]:
# Section: Data acquisition (robust)

import os, glob, shutil

RAW_CSV_PATH = CONFIG["raw_csv_path"]
print("RAW_CSV_PATH from CONFIG:", RAW_CSV_PATH)

# 1) Mount Drive if requested
if CONFIG.get("drive_mount", False):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        print("Drive mounted.")
    except Exception as e:
        print("Drive mount skipped (environment not supported):", e)

# 2) If missing, try to locate the file in Drive (common naming/folder issue)
if not os.path.exists(RAW_CSV_PATH):
    print("\nCSV not found at RAW_CSV_PATH. Searching in /content/drive/MyDrive for likely matches...")
    mydrive = "/content/drive/MyDrive"
    if os.path.exists(mydrive):
        candidates = glob.glob(mydrive + "/**/*.csv", recursive=True)
        candidates = [c for c in candidates if ("chicago" in c.lower() or "crime" in c.lower())]
        print("Matches found:", len(candidates))
        for c in candidates[:20]:
            print(" -", c)
        if len(candidates) == 1:
            RAW_CSV_PATH = candidates[0]
            CONFIG["raw_csv_path"] = RAW_CSV_PATH
            print("\n✅ Auto-selected CSV:", RAW_CSV_PATH)
    else:
        print("MyDrive path not found:", mydrive)

# 3) If still missing and allowed, download (LOCAL first) then copy into Drive path
if (not os.path.exists(RAW_CSV_PATH)) and CONFIG.get("download_if_missing", False):
    import gdown

    local_path = "/content/Chicago_Crimes.csv"
    drive_target = RAW_CSV_PATH  # keep your configured destination

    # Ensure parent dir exists (especially if drive_target is in a folder)
    os.makedirs(os.path.dirname(drive_target), exist_ok=True)

    print("\nDownloading via gdown...")
    print("File ID:", CONFIG.get("gdrive_file_id"))
    print("Local download path:", local_path)
    ret = gdown.download(id=CONFIG["gdrive_file_id"], output=local_path, quiet=False, fuzzy=True)

    print("gdown return:", ret)
    print("Local exists:", os.path.exists(local_path))
    if os.path.exists(local_path):
        print("Local size (bytes):", os.path.getsize(local_path))

        # Copy into your intended RAW_CSV_PATH (Drive or local)
        print("\nCopying local -> target path:")
        print("Target:", drive_target)
        shutil.copy2(local_path, drive_target)

# 4) Final assert + print
assert os.path.exists(RAW_CSV_PATH), f"CSV not found at: {RAW_CSV_PATH}"

print("\n✅ CSV found:", RAW_CSV_PATH)
print("Size (bytes):", os.path.getsize(RAW_CSV_PATH))

RAW_CSV_PATH from CONFIG: /content/drive/MyDrive/Chicago_Crimes.csv
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.

CSV not found at RAW_CSV_PATH. Searching in /content/drive/MyDrive for likely matches...
Matches found: 1
 - /content/drive/MyDrive/Block3_Project/Chicago_Crimes.csv

✅ Auto-selected CSV: /content/drive/MyDrive/Block3_Project/Chicago_Crimes.csv

✅ CSV found: /content/drive/MyDrive/Block3_Project/Chicago_Crimes.csv
Size (bytes): 1236254791


In [7]:
# Section: Spark session (Spark UI + event logs + GPU hooks)
os.makedirs(CONFIG['project_root'], exist_ok=True)
os.makedirs(CONFIG['event_log_dir'], exist_ok=True)
os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)

def make_spark(app_name: str) -> SparkSession:
    b = (
        SparkSession.builder
        .appName(app_name)
        .config('spark.sql.adaptive.enabled', 'true')
        .config('spark.sql.adaptive.skewJoin.enabled', 'true')
        .config('spark.sql.shuffle.partitions', str(CONFIG['shuffle_partitions']))
        .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
        .config('spark.eventLog.enabled', 'true')
        .config('spark.eventLog.dir', CONFIG['event_log_dir'])
        .config('spark.sql.ui.explainMode', 'extended')
    )
    if CONFIG['use_gpu'] == 1:
        b = (b
             .config('spark.rapids.sql.enabled', 'true')
             .config('spark.plugins', 'com.nvidia.spark.SQLPlugin'))
    s = b.getOrCreate()
    s.sparkContext.setLogLevel('WARN')
    s.sparkContext.setCheckpointDir(CONFIG['checkpoint_dir'])
    return s

spark = make_spark('Block3_ChicagoCrimes_PySpark')
print('Spark UI:', spark.sparkContext.uiWebUrl)
# Spark configuration snapshot (for report: executors/cores/memory/partitions)
conf_items = dict(spark.sparkContext.getConf().getAll())
for k in sorted([k for k in conf_items.keys() if k.startswith('spark.') and any(t in k for t in ['executor', 'driver', 'memory', 'cores', 'shuffle', 'adaptive'])]):
    print(f"{k}={conf_items[k]}")
# ---------------------------------------------------------
# Adds readable names to Jobs/Stages for screenshots.
# ---------------------------------------------------------
from contextlib import contextmanager
sc = spark.sparkContext

@contextmanager
def spark_job(desc: str):
    try:
        sc.setJobDescription(desc)
        sc.setLocalProperty("callSite.short", desc)
        yield
    finally:
        sc.setJobDescription(None)
        sc.setLocalProperty("callSite.short", None)


Spark UI: http://0a6058e6c666:4040
spark.driver.extraJavaOptions=-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-modules=jdk.incubator.vector --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED -Djdk.reflect.useDirectMethodHandle=false -Dio.netty.tryReflectionSetAccessible=true
s

## Bronze → Silver → Gold (lakehouse layout)

- **Bronze:** raw CSV ingestion with explicit schema and corrupt-record capture  
- **Silver:** typed parsing (timestamps, booleans, numerics), cleaned categories, validated keys  
- **Gold:** curated tables for monitoring, ML training/evaluation, scalability experiments, and Tableau extracts

**Partitioning rationale:** Bronze is partitioned by 'Year' and Silver/Gold are partitioned by 'event_year,event_month'. These keys match the dominant access patterns in monitoring, EDA, and temporal train/validation/test splits (time-sliced queries), reducing file scanning and shuffle work for monthly/annual rollups.

In [8]:
# Section: Explicit Bronze schema
bronze_schema = T.StructType([
    T.StructField('ID', T.LongType(), True),
    T.StructField('Case Number', T.StringType(), True),
    T.StructField('Date', T.StringType(), True),
    T.StructField('Block', T.StringType(), True),
    T.StructField('IUCR', T.StringType(), True),
    T.StructField('Primary Type', T.StringType(), True),
    T.StructField('Description', T.StringType(), True),
    T.StructField('Location Description', T.StringType(), True),
    T.StructField('Arrest', T.StringType(), True),
    T.StructField('Domestic', T.StringType(), True),
    T.StructField('Beat', T.IntegerType(), True),
    T.StructField('District', T.IntegerType(), True),
    T.StructField('Ward', T.IntegerType(), True),
    T.StructField('Community Area', T.IntegerType(), True),
    T.StructField('FBI Code', T.StringType(), True),
    T.StructField('X Coordinate', T.LongType(), True),
    T.StructField('Y Coordinate', T.LongType(), True),
    T.StructField('Year', T.IntegerType(), True),
    T.StructField('Updated On', T.StringType(), True),
    T.StructField('Latitude', T.DoubleType(), True),
    T.StructField('Longitude', T.DoubleType(), True),
    T.StructField('Location', T.StringType(), True),
    T.StructField('_corrupt_record', T.StringType(), True),
])


In [ ]:
# Section: Bronze ingestion (CSV -> DataFrame) + ingestion validation
paths = {
    'bronze_parquet': f"{CONFIG['project_root']}/bronze/chicago_crimes",
    'silver_parquet': f"{CONFIG['project_root']}/silver/chicago_crimes_typed",
    'gold_root': f"{CONFIG['project_root']}/gold",
}
os.makedirs(paths['bronze_parquet'], exist_ok=True)
os.makedirs(paths['silver_parquet'], exist_ok=True)
os.makedirs(paths['gold_root'], exist_ok=True)

bronze_df = (
    spark.read.format('csv')
    .option('header', 'true')
    .option('mode', 'PERMISSIVE')
    .option('columnNameOfCorruptRecord', '_corrupt_record')
    .option('quote', '"')
    .option('escape', '"')
    .option('nullValue', '')
    .schema(bronze_schema)
    .load(RAW_CSV_PATH)
).persist()

dq_bronze = bronze_df.agg(
    F.count('*').alias('rows_total'),
    F.sum(F.col('_corrupt_record').isNotNull().cast('int')).alias('rows_corrupt'),
    F.sum(F.col('ID').isNull().cast('int')).alias('null_id'),
    F.sum(F.col('Year').isNull().cast('int')).alias('null_year'),
    F.min('Year').alias('min_year'),
    F.max('Year').alias('max_year'),
    F.countDistinct('ID').alias('distinct_id'),
)
print('Bronze DQ:')
dq_bronze.show(truncate=False)

# Write Bronze as partitioned Parquet (fast downstream)
(bronze_df.write.mode('overwrite').partitionBy('Year').parquet(paths['bronze_parquet']))
print('Bronze parquet:', paths['bronze_parquet'])
bronze_df.unpersist()


Bronze DQ:


In [ ]:
# Section: Run manifest (data lineage)
manifest_path = f"{CONFIG['project_root']}/_meta/run_manifest_{RUN_ID}.json"
os.makedirs(os.path.dirname(manifest_path), exist_ok=True)
manifest = {
    'run_id': RUN_ID,
    'timestamp_utc': datetime.utcnow().isoformat(),
    'raw_csv_path': RAW_CSV_PATH,
    'config': CONFIG,
    'paths': paths,
    'spark_ui': spark.sparkContext.uiWebUrl,
    'spark_version': spark.version,
}
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)
print('Saved run manifest:', manifest_path)


In [ ]:
# Section: Silver typing (timestamps + booleans + cleaned columns)
bronze_in = spark.read.parquet(paths['bronze_parquet'])

silver_stage = bronze_in.select(
    F.col('ID').alias('crime_id'),
    F.trim(F.col('Case Number')).alias('case_number'),
    F.trim(F.col('Date')).alias('event_time_raw'),
    F.trim(F.col('Block')).alias('block'),
    F.trim(F.col('IUCR')).alias('iucr_code'),
    F.trim(F.col('Primary Type')).alias('primary_type'),
    F.trim(F.col('Description')).alias('description'),
    F.trim(F.col('Location Description')).alias('location_description'),
    F.trim(F.col('Arrest')).alias('arrest_raw'),
    F.trim(F.col('Domestic')).alias('domestic_raw'),
    F.col('Beat').alias('beat'),
    F.col('District').alias('district'),
    F.col('Ward').alias('ward'),
    F.col('Community Area').alias('community_area'),
    F.trim(F.col('FBI Code')).alias('fbi_code'),
    F.col('X Coordinate').alias('x_coordinate'),
    F.col('Y Coordinate').alias('y_coordinate'),
    F.col('Year').alias('year_raw'),
    F.trim(F.col('Updated On')).alias('updated_time_raw'),
    F.col('Latitude').alias('latitude'),
    F.col('Longitude').alias('longitude'),
    F.trim(F.col('Location')).alias('location_text'),
    F.col('_corrupt_record').alias('corrupt_record')
)

event_ts = F.coalesce(
    F.to_timestamp('event_time_raw', 'MM/dd/yyyy hh:mm:ss a'),
    F.to_timestamp('event_time_raw', 'MM-dd-yyyy HH:mm'),
    F.to_timestamp('event_time_raw', 'MM/dd/yyyy HH:mm'),
    F.to_timestamp('event_time_raw', 'MM-dd-yyyy HH:mm:ss'),
    F.to_timestamp('event_time_raw', 'MM/dd/yyyy HH:mm:ss'),
)
updated_ts = F.coalesce(
    F.to_timestamp('updated_time_raw', 'MM/dd/yyyy hh:mm:ss a'),
    F.to_timestamp('updated_time_raw', 'MM-dd-yyyy HH:mm'),
    F.to_timestamp('updated_time_raw', 'MM/dd/yyyy HH:mm'),
    F.to_timestamp('updated_time_raw', 'MM-dd-yyyy HH:mm:ss'),
    F.to_timestamp('updated_time_raw', 'MM/dd/yyyy HH:mm:ss'),
)

silver_stage = (silver_stage
    .withColumn('event_timestamp', event_ts)
    .withColumn('updated_timestamp', updated_ts)
    .withColumn('is_arrest', F.when(F.lower('arrest_raw')=='true', F.lit(True))
                         .when(F.lower('arrest_raw')=='false', F.lit(False))
                         .otherwise(F.lit(None).cast('boolean')))
    .withColumn('is_domestic', F.when(F.lower('domestic_raw')=='true', F.lit(True))
                           .when(F.lower('domestic_raw')=='false', F.lit(False))
                           .otherwise(F.lit(None).cast('boolean')))
    .withColumn('event_date', F.to_date('event_timestamp'))
    .withColumn('event_year', F.year('event_timestamp'))
    .withColumn('event_month', F.month('event_timestamp'))
)

# Silver validity rules
silver_valid = silver_stage.filter(
    F.col('corrupt_record').isNull() &
    F.col('crime_id').isNotNull() &
    F.col('event_timestamp').isNotNull() &
    F.col('is_arrest').isNotNull()
)

# Defensive dedup:
w = Window.partitionBy('crime_id').orderBy(F.col('updated_timestamp').desc_nulls_last(), F.col('event_timestamp').desc_nulls_last())
silver_df = (silver_valid
    .withColumn('rn', F.row_number().over(w))
    .filter(F.col('rn')==1)
    .drop('rn', 'corrupt_record')
)

(silver_df.write.mode('overwrite').partitionBy('event_year','event_month').parquet(paths['silver_parquet']))
print('Silver parquet:', paths['silver_parquet'])


## RDD parallelism (demonstration)

Most processing uses DataFrames for Catalyst/Tungsten optimizations and robust CSV options.
The following cell demonstrates **RDD-based parallel parsing** ('textFile' + 'mapPartitions') on a small sample to document RDD usage and trade-offs.


In [ ]:
# Section: RDD ingestion demo (small sample)
# This is a demonstration of parallel ingestion with RDDs.
rdd_demo_out = f"{paths['gold_root']}/rdd_demo/rdd_ingestion_sample"
os.makedirs(rdd_demo_out, exist_ok=True)

sample_lines = 500000
raw_rdd = spark.sparkContext.textFile(RAW_CSV_PATH, minPartitions=CONFIG['shuffle_partitions'])
header = raw_rdd.first()
data_rdd = raw_rdd.filter(lambda x: x != header).zipWithIndex().filter(lambda kv: kv[1] < sample_lines).keys()

import csv
from io import StringIO

def parse_part(it):
    out = []
    for line in it:
        try:
            row = next(csv.reader([line]))
            out.append(row)
        except Exception:
            continue
    return iter(out)

parsed = data_rdd.mapPartitions(parse_part)
rdd_df = spark.createDataFrame(parsed, schema=[f'c{i}' for i in range(22)])
rdd_df.write.mode('overwrite').parquet(rdd_demo_out)
print('RDD demo parquet:', rdd_demo_out)


## Feature engineering and custom Transformer

A custom Transformer creates domain-specific features (time-derived, location/block parsing, and availability flags) and keeps feature logic reusable in pipelines.


In [ ]:
# Section: Custom Transformer: crime context features
class CrimeContextFeatureTransformer(Transformer, DefaultParamsReadable, DefaultParamsWritable):
    block_col = Param(Params._dummy(), 'block_col', 'Block/address column', TypeConverters.toString)
    event_time_col = Param(Params._dummy(), 'event_time_col', 'Event timestamp column', TypeConverters.toString)
    latitude_col = Param(Params._dummy(), 'latitude_col', 'Latitude column', TypeConverters.toString)
    longitude_col = Param(Params._dummy(), 'longitude_col', 'Longitude column', TypeConverters.toString)

    def __init__(self, block_col='block', event_time_col='event_timestamp', latitude_col='latitude', longitude_col='longitude'):
        super().__init__()
        self._setDefault(block_col=block_col, event_time_col=event_time_col, latitude_col=latitude_col, longitude_col=longitude_col)
        self._set(block_col=block_col, event_time_col=event_time_col, latitude_col=latitude_col, longitude_col=longitude_col)

    def _transform(self, dataset):
        block_col = self.getOrDefault(self.block_col)
        ts_col = self.getOrDefault(self.event_time_col)
        lat_col = self.getOrDefault(self.latitude_col)
        lon_col = self.getOrDefault(self.longitude_col)

        ds = (dataset
              .withColumn('event_hour', F.hour(F.col(ts_col)))
              .withColumn('event_day_of_week', F.dayofweek(F.col(ts_col)))
              .withColumn('is_weekend', F.col('event_day_of_week').isin([1,7]))
              .withColumn('is_night', (F.col('event_hour')>=20) | (F.col('event_hour')<=5))
              .withColumn('has_geo', F.col(lat_col).isNotNull() & F.col(lon_col).isNotNull())
              .withColumn('geo_missing', ~(F.col(lat_col).isNotNull() & F.col(lon_col).isNotNull()))
        )

        # Block parsing
        block_number_raw = F.regexp_extract(F.col(block_col), r'^(\d+)', 1)
        ds = ds.withColumn('block_number_raw', F.when(block_number_raw=='', None).otherwise(block_number_raw).cast('int'))
        ds = ds.withColumn('block_number_band_100', F.when(F.col('block_number_raw').isNull(), None)
                                .otherwise((F.floor(F.col('block_number_raw')/F.lit(100))*F.lit(100)).cast('int')))
        ds = ds.withColumn('street_direction', F.regexp_extract(F.col(block_col), r'\b([NSEW])\b', 1))
        ds = ds.withColumn('street_suffix', F.regexp_extract(F.col(block_col), r'\b(AVE|ST|BLVD|RD|DR|CT|PL|LN|HWY|PKWY|TER|WAY)\b$', 1))
        return ds


In [ ]:
# Section: Build Gold ML feature table
gold_features_path = f"{paths['gold_root']}/ml_features/arrest_prediction_features"
os.makedirs(gold_features_path, exist_ok=True)

silver_in = spark.read.parquet(paths['silver_parquet'])

base = (silver_in
    .withColumn('label', F.when(F.col('is_arrest')==True, F.lit(1)).otherwise(F.lit(0)))
    .select(
        'crime_id','case_number','event_timestamp','event_date','event_year','event_month','label',
        'primary_type','description','location_description','iucr_code','fbi_code',
        'beat','district','ward','community_area','block','latitude','longitude'
    )
)

feat_builder = CrimeContextFeatureTransformer()
features_df = feat_builder.transform(base).persist()
print('Gold feature rows:', features_df.count())

(features_df.write.mode('overwrite').partitionBy('event_year','event_month').parquet(gold_features_path))
features_df.unpersist()
print('Gold features parquet:', gold_features_path)


## Exploratory analysis (EDA)

Summary tables and distributions are computed for data understanding and data-quality checks. Correlations are estimated on a sampled subset to control runtime while preserving direction and magnitude patterns.


In [ ]:
# Section: EDA: quick aggregations
eda_out = f"{paths['gold_root']}/eda"
os.makedirs(eda_out, exist_ok=True)
features_in = spark.read.parquet(gold_features_path)

by_type = (features_in.groupBy('primary_type')
           .agg(F.count('*').alias('crime_count'), F.avg('label').alias('arrest_rate'))
           .orderBy(F.col('crime_count').desc()))
by_type.write.mode('overwrite').parquet(f"{eda_out}/by_primary_type")
by_type.show(10, truncate=False)


In [ ]:
# Section: Correlation matrix + heatmap
# Uses Spark ML Correlation  and saves a long-format correlation table for Tableau.

from pyspark.ml.stat import Correlation

num_cols = ['beat','district','ward','community_area','latitude','longitude','event_hour','event_day_of_week','block_number_raw','block_number_band_100','label']

sample_df = (spark.read.parquet(gold_features_path)
    .select(*num_cols)
    .sample(False, CONFIG['corr_sample_fraction'], seed=CONFIG['seed'])
    .dropna()
)

va = VectorAssembler(inputCols=num_cols, outputCol='num_vec')
vec_df = va.transform(sample_df).select('num_vec')

corr_mat = Correlation.corr(vec_df, 'num_vec', 'pearson').collect()[0][0]  # DenseMatrix
corr_np = np.array(corr_mat.toArray())

plt.figure(figsize=(10,8))
plt.imshow(corr_np)
plt.xticks(range(len(num_cols)), num_cols, rotation=90)
plt.yticks(range(len(num_cols)), num_cols)
plt.title('Correlation Heatmap (sample, Spark pearson)')
plt.tight_layout()
plt.show()

# Save long-format correlation for Tableau
rows = []
for i, a in enumerate(num_cols):
    for j, b in enumerate(num_cols):
        rows.append({'feature_a': a, 'feature_b': b, 'corr': float(corr_np[i, j])})

spark.createDataFrame(rows).write.mode('overwrite').parquet(f"{eda_out}/correlation_long")
print('Saved:', f"{eda_out}/correlation_long")


## Temporal split — Train/Validation/Test

The dataset is split in time order (quantile cutoffs) to reduce leakage and to reflect a realistic “train on past, evaluate on future” workflow.


In [ ]:
# Section: Build q90/q95 splits
splits_root = f"{paths['gold_root']}/splits/arrest_prediction_temporal_q90_q95"
os.makedirs(splits_root, exist_ok=True)

df = (spark.read.parquet(gold_features_path)
      .filter(F.col('event_timestamp').isNotNull())
      .withColumn('event_date', F.to_date('event_timestamp')))

ts_sec = df.select(F.col('event_timestamp').cast('long').alias('ts')).approxQuantile('ts', list(CONFIG['split_quantiles']), 0.001)
q90, q95 = ts_sec[0], ts_sec[1]

val_start = spark.createDataFrame([(q90,)], ['sec']).select(F.to_date(F.from_unixtime('sec')).alias('d')).collect()[0]['d']
test_start = spark.createDataFrame([(q95,)], ['sec']).select(F.to_date(F.from_unixtime('sec')).alias('d')).collect()[0]['d']

train = df.filter(F.col('event_date') < F.lit(val_start))
val = df.filter((F.col('event_date') >= F.lit(val_start)) & (F.col('event_date') < F.lit(test_start)))
test = df.filter(F.col('event_date') >= F.lit(test_start))

train.write.mode('overwrite').partitionBy('event_year','event_month').parquet(f"{splits_root}/train")
val.write.mode('overwrite').partitionBy('event_year','event_month').parquet(f"{splits_root}/validation")
test.write.mode('overwrite').partitionBy('event_year','event_month').parquet(f"{splits_root}/test")

print('val_start:', val_start, 'test_start:', test_start)
print('rows:', train.count(), val.count(), test.count())


## Pre-processing + model selection (≥4 classifiers)

A consistent feature pipeline is applied across models (indexing/encoding, imputation, scaling). Hyperparameter tuning is performed with CrossValidator under documented compute constraints.


In [ ]:
# Section: Preprocessing (fit on train) and prepared datasets
train_df = spark.read.parquet(f"{splits_root}/train").persist()
val_df = spark.read.parquet(f"{splits_root}/validation").persist()
test_df = spark.read.parquet(f"{splits_root}/test").persist()

# Clean categorical blanks -> UNKNOWN (prevents Spark ML empty-string failure)
categorical_cols = ['primary_type','location_description','iucr_code','fbi_code','street_direction','street_suffix']
def clean_cat(df):
    out = df
    for c in categorical_cols:
        out = out.withColumn(c, F.when(F.col(c).isNull() | (F.trim(F.col(c))==''), F.lit('UNKNOWN')).otherwise(F.trim(F.col(c))))
    return out

train_df = clean_cat(train_df)
val_df = clean_cat(val_df)
test_df = clean_cat(test_df)

# Booleans -> ints
bool_cols = ['is_weekend','is_night','has_geo','geo_missing']
for c in bool_cols:
    train_df = train_df.withColumn(c, F.col(c).cast('int'))
    val_df = val_df.withColumn(c, F.col(c).cast('int'))
    test_df = test_df.withColumn(c, F.col(c).cast('int'))

# Class weights
counts = train_df.groupBy('label').count().collect()
cnt = {int(r['label']): int(r['count']) for r in counts}
tot = sum(cnt.values())
w0 = tot/(2.0*cnt[0])
w1 = tot/(2.0*cnt[1])
def add_w(df):
    return df.withColumn('class_weight', F.when(F.col('label')==1, F.lit(float(w1))).otherwise(F.lit(float(w0))))
train_df = add_w(train_df)
val_df = add_w(val_df)
test_df = add_w(test_df)

numeric_cols = ['beat','district','ward','community_area','latitude','longitude','event_hour','event_day_of_week','block_number_raw','block_number_band_100']

imputer = Imputer(inputCols=numeric_cols, outputCols=[f"{c}_imp" for c in numeric_cols])
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid='keep') for c in categorical_cols]
encoder = OneHotEncoder(inputCols=[f"{c}_idx" for c in categorical_cols], outputCols=[f"{c}_ohe" for c in categorical_cols], handleInvalid='keep')
assembler_inputs = [f"{c}_imp" for c in numeric_cols] + bool_cols + [f"{c}_ohe" for c in categorical_cols]
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol='features_raw')
scaler = StandardScaler(inputCol='features_raw', outputCol='features', withMean=False, withStd=True)

preprocess = Pipeline(stages=[imputer] + indexers + [encoder, assembler, scaler])
preprocess_model = preprocess.fit(train_df)

train_p = preprocess_model.transform(train_df).select('label','class_weight','features').checkpoint(eager=True).persist()
val_p = preprocess_model.transform(val_df).select('label','class_weight','features').checkpoint(eager=True).persist()
test_p = preprocess_model.transform(test_df).select('label','class_weight','features').checkpoint(eager=True).persist()

print('prepared rows:', train_p.count(), val_p.count(), test_p.count())


In [ ]:
# Section: Feature name mapping (robust)
# Spark's StandardScaler can drop feature metadata; this cell builds a stable feature index map.

def extract_feature_names_from_metadata(df_prepared, features_col='features'):
    meta = df_prepared.schema[features_col].metadata
    attrs = meta.get('ml_attr', {}).get('attrs', {})
    out = []
    for _, arr in attrs.items():
        for a in arr:
            out.append((int(a.get('idx', -1)), a.get('name', '')))
    out = [(i, n) for i, n in out if i >= 0 and n]
    return [name for _, name in sorted(out, key=lambda x: x[0])]

# True vector dimension (from data, not from metadata)
first_vec = train_p.select('features').head()[0]
feature_dim = int(first_vec.size) if first_vec is not None else 0

feature_names = extract_feature_names_from_metadata(train_p, 'features')

# If metadata missing, build names from pipeline inputs
if (not feature_names) or (len(feature_names) != feature_dim):
    try:
        # assembler_inputs is defined in CELL 15
        numeric_names = [c for c in assembler_inputs if c.endswith('_imp')]
        bool_names = [c for c in assembler_inputs if c in bool_cols]
        ohe_cols = [c for c in assembler_inputs if c.endswith('_ohe')]

        # find OneHotEncoderModel inside preprocess_model
        ohe_model = None
        for st in preprocess_model.stages:
            if st.__class__.__name__ == 'OneHotEncoderModel':
                ohe_model = st
                break

        ohe_dims = []
        if ohe_model is not None and hasattr(ohe_model, 'categorySizes'):
            cat_sizes = list(ohe_model.categorySizes)
            drop_last = bool(ohe_model.getDropLast()) if hasattr(ohe_model, 'getDropLast') else True
            handle_invalid = str(ohe_model.getHandleInvalid()) if hasattr(ohe_model, 'getHandleInvalid') else 'error'
            for s in cat_sizes:
                dim = int(s)
                if handle_invalid == 'keep':
                    dim += 1
                if drop_last:
                    dim = max(dim - 1, 1)
                ohe_dims.append(dim)
        else:
            # Fallback: infer dims from one row
            sample_row = preprocess_model.transform(train_df.limit(1)).select(*ohe_cols).head()
            for colname in ohe_cols:
                v = sample_row[colname]
                ohe_dims.append(int(v.size) if v is not None else 0)

        built = []
        built += [f"num__{c}" for c in numeric_names]
        built += [f"bool__{c}" for c in bool_names]
        for colname, dim in zip(ohe_cols, ohe_dims):
            built += [f"cat__{colname}__{j}" for j in range(int(dim))]

        # Use built names if they match the final vector dimension
        if len(built) == feature_dim:
            feature_names = built
        else:
            feature_names = [f"feature_{i}" for i in range(feature_dim)]
    except Exception:
        feature_names = [f"feature_{i}" for i in range(feature_dim)]

print("Feature vector size:", feature_dim)
print("Feature names available:", len(feature_names))

# Create mapping only if dimension > 0
if feature_dim > 0:
    spark.createDataFrame([(i, n) for i, n in enumerate(feature_names)], ['feature_index', 'feature_name']) \
        .write.mode('overwrite').parquet(f"{paths['gold_root']}/model_artifacts/feature_index_map")
    print("Saved feature map:", f"{paths['gold_root']}/model_artifacts/feature_index_map")
else:
    print("WARNING: feature_dim == 0. Check preprocessing pipeline outputs.")


In [ ]:
# Section: Tuning sample (stratified) + CrossValidator (5 models)
from functools import reduce
sc = spark.sparkContext
tuning_fraction = min(1.0, CONFIG['tuning_target_rows']/float(train_p.count()))
tuning = train_p.sampleBy('label', {0:tuning_fraction, 1:tuning_fraction}, seed=CONFIG['seed']).persist()
print('tuning rows:', tuning.count(), 'fraction:', tuning_fraction)

evaluator = BinaryClassificationEvaluator(labelCol='label', rawPredictionCol='rawPrediction', metricName='areaUnderROC')

models = []

# 1) Logistic Regression
lr = LogisticRegression(featuresCol='features', labelCol='label', weightCol='class_weight', maxIter=40)
lr_grid = (ParamGridBuilder().addGrid(lr.regParam,[0.01,0.1]).addGrid(lr.elasticNetParam,[0.0,1.0]).build())
models.append(('logistic_regression', lr, lr_grid))

# 2) Random Forest
rf = RandomForestClassifier(featuresCol='features', labelCol='label', weightCol='class_weight')
rf_grid = (ParamGridBuilder().addGrid(rf.numTrees,[50]).addGrid(rf.maxDepth,[10,14]).build())
models.append(('random_forest', rf, rf_grid))

# 3) Gradient Boosted Trees
gbt = GBTClassifier(featuresCol='features', labelCol='label', weightCol='class_weight')
gbt_grid = (ParamGridBuilder().addGrid(gbt.maxDepth,[3,5]).addGrid(gbt.maxIter,[20]).build())
models.append(('gbt', gbt, gbt_grid))

# 4) Decision Tree
dt = DecisionTreeClassifier(featuresCol='features', labelCol='label', weightCol='class_weight')
dt_grid = (ParamGridBuilder().addGrid(dt.maxDepth,[8,12]).build())
models.append(('decision_tree', dt, dt_grid))

# 5) Linear SVM
svm = LinearSVC(featuresCol='features', labelCol='label', weightCol='class_weight', maxIter=50)
svm_grid = (ParamGridBuilder().addGrid(svm.regParam,[0.01,0.1,1.0]).build())
models.append(('linear_svm', svm, svm_grid))

model_dir = f"{paths['gold_root']}/models/spark_cv"
metrics_dir = f"{paths['gold_root']}/model_metrics"
os.makedirs(model_dir, exist_ok=True)
os.makedirs(metrics_dir, exist_ok=True)

metric_rows = []
cm_tables = []

def roc_points_binned(df_pred, score_col, bins=2000):
    scored = df_pred.select(score_col.alias('score'), F.col('label').cast('int').alias('label'))
    binned = scored.withColumn('score_bin', (F.floor(F.col('score')*F.lit(bins))/F.lit(bins)).cast('double'))
    byb = (binned.groupBy('score_bin').agg(F.sum('label').alias('pos'), (F.count('*')-F.sum('label')).alias('neg'))
           .orderBy(F.col('score_bin').desc()))
    totals = byb.agg(F.sum('pos').alias('tp'), F.sum('neg').alias('tn')).collect()[0]
    tp = float(totals['tp']) if totals['tp'] else 1.0
    tn = float(totals['tn']) if totals['tn'] else 1.0
    w = Window.orderBy(F.col('score_bin').desc()).rowsBetween(Window.unboundedPreceding, Window.currentRow)
    roc = (byb.withColumn('cum_pos', F.sum('pos').over(w))
              .withColumn('cum_neg', F.sum('neg').over(w))
              .withColumn('tpr', F.col('cum_pos')/F.lit(tp))
              .withColumn('fpr', F.col('cum_neg')/F.lit(tn))
              .select('fpr','tpr','score_bin'))
    return roc

for name, est, grid in models:
    print('\nTraining:', name, '| grid:', len(grid))
    cv = CrossValidator(estimator=est, estimatorParamMaps=grid, evaluator=evaluator,
                        numFolds=CONFIG['cv_folds'], parallelism=CONFIG['cv_parallelism'], collectSubModels=False)
    with spark_job(f"ML: CrossValidator fit - {name}"):
        cvm = cv.fit(tuning)
    cvm_path = f"{model_dir}/{name}/cv_model"
    os.makedirs(cvm_path, exist_ok=True)
    cvm.write().overwrite().save(cvm_path)
    best = cvm.bestModel

    val_pred = best.transform(val_p)
    test_pred = best.transform(test_p)

    val_auc = evaluator.evaluate(val_pred)
    test_auc = evaluator.evaluate(test_pred)
    print('AUC val:', val_auc, '| AUC test:', test_auc)

    metric_rows.append((name, float(val_auc), float(test_auc), int(CONFIG['cv_folds']), int(len(grid)), int(CONFIG['tuning_target_rows'])))

    # Confusion matrices
    cm_tables.append(val_pred.groupBy('label','prediction').count().withColumn('model_name', F.lit(name)).withColumn('split', F.lit('validation')))
    cm_tables.append(test_pred.groupBy('label','prediction').count().withColumn('model_name', F.lit(name)).withColumn('split', F.lit('test')))

    # ROC points
    if name == 'linear_svm':
        score = vector_to_list(F.col('rawPrediction')).getItem(1)
    else:
        score = vector_to_list(F.col('probability')).getItem(1)
    roc = roc_points_binned(test_pred.select('label','rawPrediction','probability') if name!='linear_svm' else test_pred.select('label','rawPrediction'), score)
    roc.withColumn('model_name', F.lit(name)).write.mode('overwrite').parquet(f"{metrics_dir}/roc_points/{name}")

    # Feature importance / coefficients
    if name in ['random_forest','gbt','decision_tree']:
        imp = best.featureImportances
        rows = [(name, int(i), float(v)) for i,v in enumerate(imp)]
        spark.createDataFrame(rows, ['model_name','feature_index','importance']).write.mode('overwrite').parquet(f"{metrics_dir}/feature_importance/{name}")
    if name == 'logistic_regression':
        coef = best.coefficients
        rows = [(name, int(i), float(v)) for i,v in enumerate(coef)]
        spark.createDataFrame(rows, ['model_name','feature_index','coefficient']).write.mode('overwrite').parquet(f"{metrics_dir}/feature_importance/{name}")

# Save summary tables
spark.createDataFrame(metric_rows, ['model_name','auc_validation','auc_test','cv_folds','grid_size','tuning_target_rows']) \
    .write.mode('overwrite').parquet(f"{metrics_dir}/model_auc_summary")
cm_all = cm_tables[0]
for t in cm_tables[1:]:
    cm_all = cm_all.unionByName(t)
cm_all.write.mode('overwrite').parquet(f"{metrics_dir}/confusion_matrix")
print('Saved metrics:', metrics_dir)
# Consolidated tables for Tableau
try:
    roc_all = spark.read.option("recursiveFileLookup","true").parquet(f"{metrics_dir}/roc_points")
    roc_all.write.mode('overwrite').parquet(f"{metrics_dir}/roc_points_all")
except Exception as e:
    print("ROC consolidation skipped:", str(e)[:200])

try:
    fi_dirs = [p for p in glob.glob(f"{metrics_dir}/feature_importance/*") if os.path.isdir(p)]
    fi_dfs = []
    for p in fi_dirs:
        dfi = spark.read.parquet(p)
        if 'importance' not in dfi.columns:
            dfi = dfi.withColumn('importance', F.lit(None).cast('double'))
        if 'coefficient' not in dfi.columns:
            dfi = dfi.withColumn('coefficient', F.lit(None).cast('double'))
        dfi = (dfi
               .withColumn('metric', F.when(F.col('importance').isNotNull(), F.lit('importance')).otherwise(F.lit('coefficient')))
               .withColumn('value', F.coalesce(F.col('importance'), F.col('coefficient'))))
        fi_dfs.append(dfi)
    if fi_dfs:
        fi_all = reduce(lambda a,b: a.unionByName(b, allowMissingColumns=True), fi_dfs)
        fi_all.write.mode('overwrite').parquet(f"{metrics_dir}/feature_importance_all")
except Exception as e:
    print("Feature importance consolidation skipped:", str(e)[:200])

tuning.unpersist()

## MLflow logging

Model metrics and artifacts are logged to MLflow. In a local notebook environment, a file-based tracking URI is used unless a remote tracking URI is configured.


## Evaluation pack (best distributed model)

Outputs include AUC/ROC, confusion matrix, bootstrap confidence intervals (sampled), and threshold selection aligned to a simple cost model. Artifacts are written under the Gold layer for reporting and Tableau.


In [ ]:
# Section: Best model evaluation (bootstrap CI + business threshold)
from pyspark.ml.tuning import CrossValidatorModel
from pyspark.ml.evaluation import BinaryClassificationEvaluator
eval_out = f"{metrics_dir}/evaluation_pack"
os.makedirs(eval_out, exist_ok=True)

auc_tbl = spark.read.parquet(f"{metrics_dir}/model_auc_summary")
best = auc_tbl.orderBy(F.col('auc_test').desc()).first()
best_name = best['model_name']
print('Best model for evaluation:', best_name)

best_cvm = CrossValidatorModel.load(f"{model_dir}/{best_name}/cv_model")
best_model = best_cvm.bestModel

pred = best_model.transform(test_p)

# Confusion matrix counts
cm = pred.groupBy(F.col('label').cast('int').alias('label'), F.col('prediction').cast('int').alias('prediction')).count() \
        .withColumn('model_name', F.lit(best_name))
cm.write.mode('overwrite').parquet(f"{eval_out}/confusion_matrix_best")
print('Saved:', f"{eval_out}/confusion_matrix_best")

# AUC point estimate
evaluator = BinaryClassificationEvaluator(labelCol='label', rawPredictionCol='rawPrediction', metricName='areaUnderROC')
auc_point = float(evaluator.evaluate(pred.select('label','rawPrediction')))

# Bootstrap CI. Using a bounded sample of predictions, then sample-with-replacement in Spark.
bootstrap_n = int(CONFIG['bootstrap_n'])
sample_max = int(CONFIG['bootstrap_sample_max_rows'])

base = pred.select(F.col('label').cast('double').alias('label'), F.col('rawPrediction').alias('rawPrediction')) \
          .sample(False, 1.0, seed=CONFIG['seed'])

base_cnt = base.count()
frac = min(1.0, float(sample_max) / float(base_cnt if base_cnt else 1))
base_s = base.sample(False, frac, seed=CONFIG['seed']).cache()
base_s.count()
print('Bootstrap sample rows:', base_s.count())

aucs = []
for i in range(bootstrap_n):
    boot = base_s.sample(True, 1.0, seed=CONFIG['seed'] + i)
    aucs.append(float(evaluator.evaluate(boot)))

auc_df = spark.createDataFrame([(float(a),) for a in aucs], ['auc'])
ci_low, ci_high = auc_df.approxQuantile('auc', [0.025, 0.975], 0.0)

ci_tbl = spark.createDataFrame([{
    'model_name': best_name,
    'auc_point': float(auc_point),
    'bootstrap_n': int(bootstrap_n),
    'bootstrap_sample_rows': int(base_s.count()),
    'ci_low_95': float(ci_low),
    'ci_high_95': float(ci_high),
}])
ci_tbl.write.mode('overwrite').parquet(f"{eval_out}/auc_bootstrap_ci")
print('Saved:', f"{eval_out}/auc_bootstrap_ci")

base_s.unpersist()

# Business metric alignment: cost-based threshold curve using score
# For probability models use probability[1]; for SVM use rawPrediction[1] as score proxy.
if best_name == 'linear_svm':
    score = vector_to_list(F.col('rawPrediction')).getItem(1)
else:
    score = vector_to_list(F.col('probability')).getItem(1)

scored = pred.select(
    F.col('label').cast('int').alias('label'),
    score.alias('score')
)

# Threshold curve by bucketing scores (for fast computation)
bins = 200
b = scored.withColumn('thr', (F.floor(F.col('score') * F.lit(bins)) / F.lit(bins)).cast('double'))

# Costs
FP_COST = float(CONFIG['fp_cost'])
FN_COST = float(CONFIG['fn_cost'])

curve = (b.groupBy('thr')
         .agg(
             F.sum(F.when((F.col('score')>=F.col('thr')) & (F.col('label')==1), 1).otherwise(0)).alias('tp'),
             F.sum(F.when((F.col('score')>=F.col('thr')) & (F.col('label')==0), 1).otherwise(0)).alias('fp'),
             F.sum(F.when((F.col('score')<F.col('thr')) & (F.col('label')==1), 1).otherwise(0)).alias('fn'),
             F.sum(F.when((F.col('score')<F.col('thr')) & (F.col('label')==0), 1).otherwise(0)).alias('tn'),
         )
         .withColumn('expected_cost', F.lit(FP_COST)*F.col('fp') + F.lit(FN_COST)*F.col('fn'))
         .orderBy('thr'))

curve.write.mode('overwrite').parquet(f"{eval_out}/cost_threshold_curve")
best_thr = curve.orderBy(F.col('expected_cost').asc()).first()
spark.createDataFrame([best_thr.asDict()]).write.mode('overwrite').parquet(f"{eval_out}/best_cost_threshold")
print('Saved:', f"{eval_out}/cost_threshold_curve")
print('Saved:', f"{eval_out}/best_cost_threshold")


In [ ]:
# Section: MLflow logging
mlflow.set_tracking_uri(CONFIG.get('mlflow_tracking_uri') or 'file:/content/mlruns')
mlflow.set_experiment('block3_chicago_crimes_pyspark')

metrics_path = f"{metrics_dir}/model_auc_summary"
auc_df = spark.read.parquet(metrics_path)
best_row = auc_df.orderBy(F.col('auc_test').desc()).first()
best_model_name = best_row['model_name']

with mlflow.start_run(run_name=f"spark_models_{RUN_ID}"):
    mlflow.log_param('cv_folds', CONFIG['cv_folds'])
    mlflow.log_param('tuning_target_rows', CONFIG['tuning_target_rows'])
    mlflow.log_metric('best_auc_test', float(best_row['auc_test']))
    mlflow.log_metric('best_auc_validation', float(best_row['auc_validation']))
    mlflow.log_param('best_model', best_model_name)

    # Log AUC summary as a single CSV artifact (Spark write -> file)
    tmp_dir = f"/content/auc_summary_{RUN_ID}"
    auc_df.coalesce(1).write.mode("overwrite").option("header", True).csv(tmp_dir)

    part_files = glob.glob(tmp_dir + "/part-*.csv")
    if part_files:
        mlflow.log_artifact(part_files[0], artifact_path="tables")

print('Best Spark model:', best_model_name)


## Single-node baseline (scikit-learn) + serialization

A scikit-learn baseline is trained on a stratified sample from the training split to provide a single-node comparison and to demonstrate Pickle-based serialization. Distributed training remains the primary result.


In [ ]:
# Section: Build and save sklearn baseline sample
baseline_dir = f"{paths['gold_root']}/baseline_sklearn"
os.makedirs(baseline_dir, exist_ok=True)

baseline_target = 200000
train_rows = train_df.count()
frac = min(1.0, baseline_target/float(train_rows))
baseline_sample = train_df.sampleBy('label', {0:frac,1:frac}, seed=CONFIG['seed']).persist()
print('baseline rows:', baseline_sample.count())
baseline_sample.write.mode('overwrite').parquet(f"{baseline_dir}/train_sample")
baseline_sample.unpersist()
print('Saved:', f"{baseline_dir}/train_sample")


In [ ]:
# Section: scikit-learn baseline + Pickle serialization
# Main pipeline + all primary models are PySpark MLlib.
# This sklearn baseline is required for rubric "single-node comparison".

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression as SkLogReg
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve
import pickle

sample_sdf = spark.read.parquet(f"{baseline_dir}/train_sample").select(
    'label',
    'beat','district','ward','community_area','latitude','longitude',
    'event_hour','event_day_of_week','block_number_raw','block_number_band_100',
    'is_weekend','is_night','has_geo','geo_missing',
    'primary_type','location_description','iucr_code','fbi_code','street_direction','street_suffix'
)

# Normalize categoricals and booleans in Spark first
cat_cols = ['primary_type','location_description','iucr_code','fbi_code','street_direction','street_suffix']
for c in cat_cols:
    sample_sdf = sample_sdf.withColumn(c, F.when(F.col(c).isNull() | (F.trim(F.col(c))==''), F.lit('UNKNOWN')).otherwise(F.col(c)))

bool_cols = ['is_weekend','is_night','has_geo','geo_missing']
for c in bool_cols:
    sample_sdf = sample_sdf.withColumn(c, F.col(c).cast('int'))

num_cols = ['beat','district','ward','community_area','latitude','longitude','event_hour','event_day_of_week','block_number_raw','block_number_band_100']

# Median map for numeric imputation (computed on sample, collected to driver)
median_map = {}
for c in num_cols:
    try:
        m = sample_sdf.approxQuantile(c, [0.5], 0.001)
        median_map[c] = float(m[0]) if m and m[0] is not None else 0.0
    except Exception:
        median_map[c] = 0.0

X_dicts = []
y = []

for r in sample_sdf.toLocalIterator():
    rd = r.asDict()
    y.append(int(rd['label']))
    d = {}
    for c in num_cols:
        v = rd.get(c)
        d[c] = float(v) if v is not None else float(median_map[c])
    for c in bool_cols:
        v = rd.get(c)
        d[c] = int(v) if v is not None else 0
    for c in cat_cols:
        v = rd.get(c)
        d[c] = str(v) if v not in (None, '') else 'UNKNOWN'
    X_dicts.append(d)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_dicts, y, test_size=0.2, random_state=CONFIG['seed'], stratify=y
)

vec = DictVectorizer(sparse=True)
Xtr = vec.fit_transform(X_tr)
Xte = vec.transform(X_te)

clf = SkLogReg(max_iter=200, class_weight='balanced', n_jobs=-1)
clf.fit(Xtr, y_tr)

proba = clf.predict_proba(Xte)[:, 1]
pred = (proba >= 0.5).astype(int)

auc = float(roc_auc_score(y_te, proba))
cm = confusion_matrix(y_te, pred)
fpr, tpr, thr = roc_curve(y_te, proba)

print('sklearn baseline AUC:', auc)
print('sklearn confusion matrix:\n', cm)

# Save baseline outputs to Gold (Tableau/report)
sk_out = f"{baseline_dir}/sklearn_baseline"
os.makedirs(sk_out, exist_ok=True)

spark.createDataFrame([{
    'model_name': 'sklearn_logistic_regression',
    'auc_test': float(auc),
    'rows_test': int(len(y_te))
}]).write.mode('overwrite').parquet(f"{sk_out}/metrics")

spark.createDataFrame([{
    'model_name': 'sklearn_logistic_regression',
    'threshold': 0.5,
    'tn': int(cm[0,0]), 'fp': int(cm[0,1]),
    'fn': int(cm[1,0]), 'tp': int(cm[1,1]),
}]).write.mode('overwrite').parquet(f"{sk_out}/confusion_matrix")

# Downsample ROC points for Tableau if very long
max_pts = 2000
idx = np.linspace(0, len(fpr)-1, num=min(max_pts, len(fpr)), dtype=int)
roc_rows = [{'model_name':'sklearn_logistic_regression','fpr': float(fpr[i]), 'tpr': float(tpr[i]), 'threshold': float(thr[i])} for i in idx]
spark.createDataFrame(roc_rows).write.mode('overwrite').parquet(f"{sk_out}/roc_points")

# Serialize baseline (pickle) — required by rubric
pkl_path = f"/content/sklearn_baseline_{RUN_ID}.pkl"
with open(pkl_path, "wb") as f:
    pickle.dump({'vectorizer': vec, 'model': clf}, f)

print('Saved pickle:', pkl_path)


In [ ]:
# Section: Explainability (Spark-only approximations): LIME-like + SHAP-like
# Uses: preprocess_model (raw -> features) and the best Spark model from evaluation cell.

from pyspark.ml.tuning import CrossValidatorModel
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import VectorAssembler
explain_out = f"{paths['gold_root']}/model_metrics/explainability_spark_only"
os.makedirs(explain_out, exist_ok=True)

# Load best model again (consistent with evaluation pack)
best_name = spark.read.parquet(f"{metrics_dir}/model_auc_summary").orderBy(F.col('auc_test').desc()).first()['model_name']
best_cvm = CrossValidatorModel.load(f"{model_dir}/{best_name}/cv_model")
best_model = best_cvm.bestModel

# Predictor on raw rows: raw -> features -> model -> score
def predict_scores(raw_df):
    prepared = preprocess_model.transform(raw_df)
    scored = best_model.transform(prepared)
    if 'probability' in scored.columns:
        s = vector_to_list(F.col('probability')).getItem(1)
    else:
        s = vector_to_list(F.col('rawPrediction')).getItem(1)
    return scored.select(s.alias('score'))

# Choose numeric columns for local surrogate (keep small)
lime_numeric = ['beat','district','ward','community_area','latitude','longitude','event_hour','event_day_of_week','block_number_raw','block_number_band_100']
lime_numeric = [c for c in lime_numeric if c in test_df.columns]

# Base point x0 from test (non-null)
row = test_df.select(*lime_numeric).dropna().limit(1).collect()
if not row:
    print('No suitable row for LIME-like; skipping.')
    lime_df = spark.createDataFrame([], 'feature string, weight double')
else:
    x0 = row[0].asDict()
    stats = test_df.select(*[F.stddev(F.col(c)).alias(c) for c in lime_numeric]).collect()[0].asDict()
    stds = {c: float(stats.get(c) if stats.get(c) not in (None, 0.0) else 1.0) for c in lime_numeric}

    p = spark.range(0, CONFIG['lime_num_perturb']).select(F.lit(1).alias('dummy'))
    for c in lime_numeric:
        p = p.withColumn(c, F.lit(float(x0[c])) + F.randn(CONFIG['seed'] + abs(hash(c)) % 10000) * F.lit(stds[c] * CONFIG['lime_sigma_frac']))

    # Keep categoricals/constants from an arbitrary base row
    keep_cols = [c for c in test_df.columns if c not in lime_numeric]
    base_row = test_df.select(*keep_cols).limit(1).collect()[0].asDict()
    for c in keep_cols:
        p = p.withColumn(c, F.lit(base_row[c]))

    scores = predict_scores(p).withColumn('score', F.col('score').cast('double'))
    # distance & kernel weights
    dist = None
    for c in lime_numeric:
        term = (F.col(c) - F.lit(float(x0[c])))**2
        dist = term if dist is None else dist + term
    ddf = p.join(scores).withColumn('dist', F.sqrt(dist)).withColumn('w', F.exp(-(F.col('dist')**2) / F.lit(CONFIG['lime_kernel_width']**2)))

    assembler = VectorAssembler(inputCols=lime_numeric, outputCol='lime_features', handleInvalid='keep')

    # LIME-like local surrogate for a classifier:
    # - Use the black-box model's predicted class as the surrogate label (0/1).
    # - Fit a weighted LogisticRegression locally to approximate the decision boundary.
    dfeat = (
        assembler.transform(ddf)
        .select(
            'lime_features',
            (F.col('score') >= F.lit(0.5)).cast('double').alias('label'),
            F.col('w').cast('double').alias('w')
        )
        .na.drop(subset=['lime_features', 'label', 'w'])
    )

    # Guard: need non-empty neighborhood and both classes present
    if dfeat.rdd.isEmpty() or dfeat.select('label').distinct().count() < 2:
        print('LIME-like: insufficient class variation in neighborhood; skipping local surrogate fit.')
        lime_df = spark.createDataFrame([], 'feature string, weight double, abs_weight double')
    else:
        local = LogisticRegression(
            featuresCol='lime_features',
            labelCol='label',
            weightCol='w',
            maxIter=50
        ).fit(dfeat)

        coefs = list(local.coefficients)
        lime_df = spark.createDataFrame(
            [(lime_numeric[i], float(coefs[i])) for i in range(len(lime_numeric))],
            ['feature','weight']
        ).withColumn('abs_weight', F.abs('weight')).orderBy(F.desc('abs_weight'))

lime_df.write.mode('overwrite').parquet(f"{explain_out}/lime_like")
print('Saved:', f"{explain_out}/lime_like")

# SHAP-like (Monte Carlo) on top-k numeric features (small, bounded)
shap_cols = lime_numeric[:min(6, len(lime_numeric))]
if not shap_cols:
    shap_df = spark.createDataFrame([], 'feature string, shap_value double')
else:
    # Baseline as mean of features
    base = test_df.select(*shap_cols).dropna().sample(False, 0.02, seed=CONFIG['seed']).cache()
    base_row = base.agg(*[F.avg(c).alias(c) for c in shap_cols]).collect()[0].asDict()
    baseline = {c: float(base_row[c] if base_row[c] is not None else 0.0) for c in shap_cols}

    sample = base.orderBy(F.rand(CONFIG['seed'])).limit(CONFIG['shap_samples']).cache()
    rows = [r.asDict() for r in sample.collect()]
    rnd = random.Random(CONFIG['seed'])

    def score_batch(dict_rows):
        sdf = spark.createDataFrame([Row(**d) for d in dict_rows])
        # attach required non-shap columns from a single base row
        base_other = test_df.drop(*shap_cols).limit(1)
        sdf = sdf.crossJoin(base_other)
        return [float(x['s']) for x in predict_scores(sdf).select(F.col('score').alias('s')).collect()]

    shap_acc = {c: 0.0 for c in shap_cols}
    denom = 0
    for r in rows:
        for _ in range(CONFIG['shap_mc_per_row']):
            order = shap_cols[:]
            rnd.shuffle(order)
            coalition = []
            cur = dict(baseline)
            coalition.append(dict(cur))
            for feat in order:
                cur = dict(cur)
                cur[feat] = float(r[feat])
                coalition.append(dict(cur))
            preds = score_batch(coalition)
            for j, feat in enumerate(order):
                shap_acc[feat] += (preds[j+1] - preds[j])
            denom += 1

    out = [(c, shap_acc[c]/float(denom) if denom else 0.0) for c in shap_cols]
    shap_df = spark.createDataFrame(out, ['feature','shap_value']).withColumn('abs_shap', F.abs('shap_value')).orderBy(F.desc('abs_shap'))

shap_df.write.mode('overwrite').parquet(f"{explain_out}/shap_like")
print('Saved:', f"{explain_out}/shap_like")


## Performance optimization evidence

Benchmarks and diagnostic outputs are produced to support broadcast joins, caching/persist strategy, shuffle/partition tuning, and Spark UI evidence.


In [ ]:
sc = spark.sparkContext
# Section: Optimization benchmarks
bench_root = f"{paths['gold_root']}/optimization/benchmarks"
os.makedirs(bench_root, exist_ok=True)

train_base = spark.read.parquet(f"{splits_root}/train").select('crime_id','fbi_code','primary_type','label','event_year','event_month').persist()
train_n = train_base.count()

# lookup (tiny)
mode_counts = train_base.groupBy('fbi_code','primary_type').agg(F.count('*').alias('cnt'))
rn = F.row_number().over(Window.partitionBy('fbi_code').orderBy(F.col('cnt').desc()))
lookup = (mode_counts.withColumn('rn', rn).filter(F.col('rn')==1)
          .select(F.col('fbi_code').alias('fbi_code_key'), F.col('primary_type').alias('fbi_primary_type_mode')))
lookup_n = lookup.count()

def timed_count(df, label):
    sc.setJobDescription(f"Optimization: {label}")
    t0 = time.perf_counter(); n = df.count(); dt = time.perf_counter()-t0
    print(label, '| rows=', n, '| sec=', round(dt,2))
    sc.setJobDescription(None)
    return float(dt)

# shuffle join baseline
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', -1)
shuffle_join = train_base.join(lookup, train_base['fbi_code']==lookup['fbi_code_key'], 'left')
t_shuffle = timed_count(shuffle_join, 'shuffle_join')

# broadcast join
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', 10485760)
bcast_join = train_base.join(F.broadcast(lookup), train_base['fbi_code']==lookup['fbi_code_key'], 'left')
t_bcast = timed_count(bcast_join, 'broadcast_join')

spark.createDataFrame([{
    'train_rows': int(train_n), 'lookup_rows': int(lookup_n),
    'seconds_shuffle': float(t_shuffle), 'seconds_broadcast': float(t_bcast),
    'speedup_ratio': float(t_shuffle/t_bcast) if t_bcast>0 else None
}]).write.mode('overwrite').parquet(f"{bench_root}/broadcast_join_benchmark")

# cache benchmark
agg = train_base.groupBy('event_year','event_month','primary_type').agg(F.count('*').alias('crime_count'), F.avg('label').alias('arrest_rate'))
t1 = timed_count(agg, 'agg_no_cache_run1')
t2 = timed_count(agg, 'agg_no_cache_run2')
agg_c = agg.persist()
t3 = timed_count(agg_c, 'agg_cached_run1')
t4 = timed_count(agg_c, 'agg_cached_run2')
agg_c.unpersist()
spark.createDataFrame([{
    'no_cache_run1': t1, 'no_cache_run2': t2, 'cache_run1': t3, 'cache_run2': t4,
    'speedup_second_run': float(t2/t4) if t4>0 else None
}]).write.mode('overwrite').parquet(f"{bench_root}/cache_benchmark")

# shuffle partitions benchmark
results = []
for p in [50,200,800]:
    spark.conf.set('spark.sql.shuffle.partitions', int(p))
    dt = timed_count(agg, f'shuffle_partitions={p}')
    results.append({'shuffle_partitions': int(p), 'seconds': float(dt)})
spark.createDataFrame(results).write.mode('overwrite').parquet(f"{bench_root}/shuffle_partitions_benchmark")

train_base.unpersist()
print('Saved benchmarks:', bench_root)

## Scalability analysis (strong and weak scaling)

- **Strong scaling:** fixed problem size, vary parallelism/partitions and measure runtime  
- **Weak scaling:** increase problem size with resources and measure throughput  

Results are saved in Gold for Tableau and the report.

**Environment note:** In Google Colab a true multi-node Spark cluster is not available. Strong/weak scaling is therefore proxied using effective parallelism controls (e.g., shuffle partitions) and workload size scaling. Report conclusions explicitly under these constraints.

In [ ]:
# Section: Strong + weak scaling tables
scaling_root = f"{paths['gold_root']}/scalability"
os.makedirs(scaling_root, exist_ok=True)
strong_out = f"{scaling_root}/strong_scaling"
weak_out = f"{scaling_root}/weak_scaling"

base_df = spark.read.parquet(f"{splits_root}/train").select('crime_id','fbi_code','primary_type','label')
mode_counts = base_df.groupBy('fbi_code','primary_type').agg(F.count('*').alias('cnt'))
rn = F.row_number().over(Window.partitionBy('fbi_code').orderBy(F.col('cnt').desc()))
lookup = (mode_counts.withColumn('rn', rn).filter(F.col('rn')==1)
          .select(F.col('fbi_code').alias('fbi_code_key'), F.col('primary_type').alias('fbi_primary_type_mode')))
lookup_n = lookup.count()

def workload(df_in, shuffle_partitions):
    spark.conf.set('spark.sql.shuffle.partitions', int(shuffle_partitions))
    joined = df_in.join(F.broadcast(lookup), df_in['fbi_code']==lookup['fbi_code_key'], 'left')
    agg = joined.groupBy('fbi_code','fbi_primary_type_mode').agg(F.count('*').alias('rows'), F.avg('label').alias('arrest_rate'))
    t0 = time.perf_counter(); out = agg.count(); dt = time.perf_counter()-t0
    return float(dt), int(out)

# Strong scaling
fixed = base_df.sample(False, 0.2, seed=CONFIG['seed']).persist()
fixed_rows = fixed.count()
strong = []
for p in [50,100,200,400]:
    sec, outn = workload(fixed, p)
    strong.append({'rows_in': int(fixed_rows), 'shuffle_partitions': int(p), 'seconds': sec,
                   'throughput_rows_per_sec': float(fixed_rows/sec) if sec>0 else None,
                   'lookup_rows': int(lookup_n)})
fixed.unpersist()
spark.createDataFrame(strong).write.mode('overwrite').parquet(strong_out)

# Weak scaling
weak_cfg = [(0.05,50),(0.10,100),(0.20,200),(0.40,400)]
weak = []
for frac, p in weak_cfg:
    df = base_df.sample(False, frac, seed=CONFIG['seed']).persist()
    n = df.count()
    sec, outn = workload(df, p)
    weak.append({'fraction': float(frac), 'rows_in': int(n), 'shuffle_partitions': int(p), 'seconds': sec,
                 'throughput_rows_per_sec': float(n/sec) if sec>0 else None,
                 'lookup_rows': int(lookup_n)})
    df.unpersist()
spark.createDataFrame(weak).write.mode('overwrite').parquet(weak_out)

print('Saved scaling:', scaling_root)


## Cost–performance trade-off

Derived metrics (e.g., seconds per million rows and throughput) support a cost–performance discussion and a practical recommendation section.


In [ ]:
# Section: Cost/performance derived metrics
cost_out = f"{paths['gold_root']}/scalability/cost_performance"
os.makedirs(cost_out, exist_ok=True)

strong = spark.read.parquet(f"{paths['gold_root']}/scalability/strong_scaling")
weak = spark.read.parquet(f"{paths['gold_root']}/scalability/weak_scaling")

def add_cost(df, label):
    return (df
        .withColumn('scenario', F.lit(label))
        .withColumn('sec_per_million_rows', (F.col('seconds')/F.col('rows_in'))*F.lit(1e6))
    )

strong_c = add_cost(strong, 'strong')
weak_c = add_cost(weak, 'weak')
combined = strong_c.unionByName(weak_c, allowMissingColumns=True)
combined.write.mode('overwrite').parquet(cost_out)
print('Saved:', cost_out)


## Tableau extracts (Dashboards 1–4) + export manifest

This section writes Tableau-ready tables and a manifest with file paths. Dashboards are built from these curated extracts:
1) data quality & pipeline monitoring, 2) model performance & feature importance, 3) business insights, 4) scalability & cost.

After the Parquet extracts are written, the next cell exports each dataset to **single-file CSV** folders and packages them into one **ZIP** for download from Colab.

In [ ]:
# Section: Dashboard datasets + manifest (Parquet) + consolidated model tables
tableau_root = f"{paths['gold_root']}/tableau"
dash1_root = f"{tableau_root}/dashboard_1_data_quality"
dash2_root = f"{tableau_root}/dashboard_2_model_performance"
dash3_root = f"{tableau_root}/dashboard_3_business_insights"
dash4_root = f"{tableau_root}/dashboard_4_scalability_cost"
os.makedirs(dash1_root, exist_ok=True)
os.makedirs(dash2_root, exist_ok=True)
os.makedirs(dash3_root, exist_ok=True)
os.makedirs(dash4_root, exist_ok=True)

silver_df = spark.read.parquet(paths['silver_parquet']).persist()

# -------------------------
# Dashboard 1: Data quality
# -------------------------
with spark_job("Tableau D1: DQ overall + null profile + monthly volume"):
    dq_overall = (silver_df.agg(
        F.count('*').alias('rows_silver'),
        F.min('event_timestamp').alias('min_event_timestamp'),
        F.max('event_timestamp').alias('max_event_timestamp'),
        F.sum(F.col('latitude').isNull().cast('int')).alias('null_latitude'),
        F.sum(F.col('longitude').isNull().cast('int')).alias('null_longitude')
    ))
    dq_overall.write.mode('overwrite').parquet(f"{dash1_root}/dq_overall")

    # Layer row counts (Bronze/Silver/Gold)
    layer_rows = []
    layer_map = {
        "bronze": paths['bronze_parquet'],
        "silver": paths['silver_parquet'],
        "gold_features": globals().get("gold_features_path", f"{paths['gold_root']}/ml_features/arrest_prediction_features"),
    }
    for layer, p in layer_map.items():
        if os.path.exists(p):
            try:
                layer_rows.append((layer, p, int(spark.read.parquet(p).count())))
            except Exception:
                layer_rows.append((layer, p, None))
        else:
            layer_rows.append((layer, p, None))
    spark.createDataFrame(layer_rows, ['layer','path','row_count']).write.mode('overwrite').parquet(f"{dash1_root}/layer_row_counts")

    # Null profile (stack-based, no driver collect)
    total_rows = int(silver_df.count())
    cols = silver_df.columns
    null_exprs = [F.sum(F.col(c).isNull().cast('int')).alias(c) for c in cols]
    null_counts = silver_df.agg(*null_exprs)
    n = len(cols)
    stack_args = ", ".join([f"'{c}', `{c}`" for c in cols])
    stack_expr = f"stack({n}, {stack_args}) as (column_name, null_count)"
    null_profile = (null_counts.selectExpr(stack_expr)
                    .withColumn('rows_total', F.lit(total_rows))
                    .withColumn('null_rate', F.when(F.col('rows_total')>0, F.col('null_count')/F.col('rows_total')).otherwise(F.lit(None)))
                    .orderBy(F.desc('null_rate')))
    null_profile.write.mode('overwrite').parquet(f"{dash1_root}/null_profile_by_column")

    # Monthly volume
    monthly = (silver_df.groupBy('event_year','event_month')
               .agg(F.count('*').alias('row_count'))
               .orderBy('event_year','event_month'))
    monthly.write.mode('overwrite').parquet(f"{dash1_root}/volume_by_month")

# ----------------------------------------
# Dashboard 2: Model performance (Spark + sklearn baseline)
# ----------------------------------------
with spark_job("Tableau D2: consolidate model metrics"):
    # Spark ML tables
    spark.read.parquet(f"{metrics_dir}/model_auc_summary").write.mode('overwrite').parquet(f"{dash2_root}/model_auc_summary")
    spark.read.parquet(f"{metrics_dir}/confusion_matrix").write.mode('overwrite').parquet(f"{dash2_root}/confusion_matrix")

    # Prefer consolidated paths if available
    roc_src = f"{metrics_dir}/roc_points_all" if os.path.exists(f"{metrics_dir}/roc_points_all") else f"{metrics_dir}/roc_points"
    fi_src  = f"{metrics_dir}/feature_importance_all" if os.path.exists(f"{metrics_dir}/feature_importance_all") else f"{metrics_dir}/feature_importance"

    roc_all = spark.read.option("recursiveFileLookup","true").parquet(roc_src)
    roc_all.write.mode('overwrite').parquet(f"{dash2_root}/roc_points_all")

    # Feature importance: ensure unified metric/value columns
    if os.path.exists(f"{metrics_dir}/feature_importance_all"):
        fi_all = spark.read.parquet(f"{metrics_dir}/feature_importance_all")
    else:
        fi_dirs = [p for p in glob.glob(f"{fi_src}/*") if os.path.isdir(p)]
        fi_dfs = []
        for p in fi_dirs:
            dfi = spark.read.parquet(p)
            if 'importance' not in dfi.columns:
                dfi = dfi.withColumn('importance', F.lit(None).cast('double'))
            if 'coefficient' not in dfi.columns:
                dfi = dfi.withColumn('coefficient', F.lit(None).cast('double'))
            dfi = (dfi
                   .withColumn('metric', F.when(F.col('importance').isNotNull(), F.lit('importance')).otherwise(F.lit('coefficient')))
                   .withColumn('value', F.coalesce(F.col('importance'), F.col('coefficient'))))
            fi_dfs.append(dfi)
        fi_all = fi_dfs[0]
        for d in fi_dfs[1:]:
            fi_all = fi_all.unionByName(d, allowMissingColumns=True)
    fi_all.write.mode('overwrite').parquet(f"{dash2_root}/feature_importance_all")

    # sklearn baseline (if present)
    try:
        sk_out = globals().get("sk_out", f"{baseline_dir}/sklearn_baseline")
        if os.path.exists(sk_out):
            if os.path.exists(f"{sk_out}/metrics_summary"):
                spark.read.parquet(f"{sk_out}/metrics_summary").write.mode('overwrite').parquet(f"{dash2_root}/sklearn_metrics_summary")
            if os.path.exists(f"{sk_out}/confusion_matrix"):
                spark.read.parquet(f"{sk_out}/confusion_matrix").write.mode('overwrite').parquet(f"{dash2_root}/sklearn_confusion_matrix")
            if os.path.exists(f"{sk_out}/roc_points"):
                spark.read.parquet(f"{sk_out}/roc_points").write.mode('overwrite').parquet(f"{dash2_root}/sklearn_roc_points")
    except Exception as e:
        print("sklearn baseline copy skipped:", str(e)[:200])

# ------------------------------
# Dashboard 3: Business insights
# ------------------------------
with spark_job("Tableau D3: business insights aggregates"):
    core = (silver_df
        .withColumn('label', F.when(F.col('is_arrest')==True, F.lit(1)).otherwise(F.lit(0)))
        .withColumn('event_hour', F.hour('event_timestamp'))
        .withColumn('event_day_of_week', F.dayofweek('event_timestamp'))
    )

    core.groupBy('event_year','event_month','event_day_of_week','event_hour')         .agg(F.count('*').alias('crime_count'), F.avg('label').alias('arrest_rate'))         .write.mode('overwrite').parquet(f"{dash3_root}/time_volume_arrest_rate")

    core.groupBy('event_year','district','ward','community_area','primary_type')         .agg(F.count('*').alias('crime_count'), F.avg('label').alias('arrest_rate'))         .write.mode('overwrite').parquet(f"{dash3_root}/geo_hotspots_by_type")

    core.groupBy('event_year','location_description','primary_type')         .agg(F.count('*').alias('crime_count'), F.avg('label').alias('arrest_rate'))         .write.mode('overwrite').parquet(f"{dash3_root}/location_context")

    core.groupBy('event_year','primary_type')         .agg(F.count('*').alias('crime_count'))         .write.mode('overwrite').parquet(f"{dash3_root}/primary_type_trends")

    # Optional: monthly trends by district (if columns exist)
    if 'district' in core.columns:
        (core.groupBy('event_year','event_month','district')
             .agg(F.count('*').alias('crime_count'), F.avg('label').alias('arrest_rate'))
             .write.mode('overwrite').parquet(f"{dash3_root}/by_district_month"))

# ---------------------------------
# Dashboard 4: Scalability and cost
# ---------------------------------
with spark_job("Tableau D4: optimization + scaling tables"):
    # Optimization benchmarks
    for name in ['broadcast_join_benchmark','cache_benchmark','shuffle_partitions_benchmark']:
        p = f"{bench_root}/{name}"
        if os.path.exists(p):
            spark.read.parquet(p).write.mode('overwrite').parquet(f"{dash4_root}/{name}")

    # Scaling tables
    for name in ['strong_scaling','weak_scaling','cost_performance']:
        p = f"{paths['gold_root']}/scalability/{name}"
        if os.path.exists(p):
            spark.read.parquet(p).write.mode('overwrite').parquet(f"{dash4_root}/{name}")

# Manifest (Parquet extract paths)
manifest_rows = [
    ('Dashboard 1','dq_overall', f"{dash1_root}/dq_overall"),
    ('Dashboard 1','layer_row_counts', f"{dash1_root}/layer_row_counts"),
    ('Dashboard 1','null_profile_by_column', f"{dash1_root}/null_profile_by_column"),
    ('Dashboard 1','volume_by_month', f"{dash1_root}/volume_by_month"),

    ('Dashboard 2','model_auc_summary', f"{dash2_root}/model_auc_summary"),
    ('Dashboard 2','confusion_matrix', f"{dash2_root}/confusion_matrix"),
    ('Dashboard 2','roc_points_all', f"{dash2_root}/roc_points_all"),
    ('Dashboard 2','feature_importance_all', f"{dash2_root}/feature_importance_all"),
    ('Dashboard 2','sklearn_metrics_summary', f"{dash2_root}/sklearn_metrics_summary"),
    ('Dashboard 2','sklearn_confusion_matrix', f"{dash2_root}/sklearn_confusion_matrix"),
    ('Dashboard 2','sklearn_roc_points', f"{dash2_root}/sklearn_roc_points"),

    ('Dashboard 3','time_volume_arrest_rate', f"{dash3_root}/time_volume_arrest_rate"),
    ('Dashboard 3','geo_hotspots_by_type', f"{dash3_root}/geo_hotspots_by_type"),
    ('Dashboard 3','location_context', f"{dash3_root}/location_context"),
    ('Dashboard 3','primary_type_trends', f"{dash3_root}/primary_type_trends"),
    ('Dashboard 3','by_district_month', f"{dash3_root}/by_district_month"),

    ('Dashboard 4','broadcast_join_benchmark', f"{dash4_root}/broadcast_join_benchmark"),
    ('Dashboard 4','cache_benchmark', f"{dash4_root}/cache_benchmark"),
    ('Dashboard 4','shuffle_partitions_benchmark', f"{dash4_root}/shuffle_partitions_benchmark"),
    ('Dashboard 4','strong_scaling', f"{dash4_root}/strong_scaling"),
    ('Dashboard 4','weak_scaling', f"{dash4_root}/weak_scaling"),
    ('Dashboard 4','cost_performance', f"{dash4_root}/cost_performance"),
]
spark.createDataFrame(manifest_rows, ['dashboard','dataset_name','path']).write.mode('overwrite').parquet(f"{tableau_root}/export_manifest")
print('Saved manifest:', f"{tableau_root}/export_manifest")

silver_df.unpersist()


In [ ]:
# Section: Export Tableau extracts to CSV + ZIP (download)
import shutil

tableau_root = f"{paths['gold_root']}/tableau"
csv_root = f"{tableau_root}/csv"
os.makedirs(csv_root, exist_ok=True)

def export_parquet_to_csv(parquet_path: str, csv_out: str, coalesce_to: int = 1):
    if not os.path.exists(parquet_path):
        print("Skip (missing):", parquet_path)
        return
    df = spark.read.option("recursiveFileLookup","true").parquet(parquet_path)
    (df.coalesce(coalesce_to)
       .write.mode("overwrite")
       .option("header", True)
       .csv(csv_out))
    print("CSV ->", csv_out)

# Read manifest and export each dataset
manifest_df = spark.read.parquet(f"{tableau_root}/export_manifest")
manifest = [(r['dashboard'], r['dataset_name'], r['path']) for r in manifest_df.collect()]

for dash, name, p in manifest:
    dash_slug = dash.lower().replace(" ", "_")
    out_dir = f"{csv_root}/{dash_slug}/{name}"
    export_parquet_to_csv(p, out_dir)

# Create a single ZIP for download/sharing
zip_base = f"{tableau_root}/tableau_csv_exports"
zip_path = f"{zip_base}.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)
shutil.make_archive(zip_base, "zip", csv_root)
print("ZIP created:", zip_path)

# Download in Colab (optional)
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Download skipped (not running in Colab UI):", e)


## Spark UI screenshots (evidence checklist)

Capture Spark UI evidence during the following runs and include it in the report:
- Ingestion + Parquet write (Jobs/Stages)
- Broadcast join benchmark (SQL tab plan + physical operator)
- Cache/persist demonstration (Storage tab)
- Tuning run (Stages tab: shuffles, spills, task skew)
- Scalability runs (Jobs/Stages + Environment for config)

Recommended tabs: **Jobs, Stages, SQL, Storage, Executors, Environment**.


In [ ]:
# Section: Open Spark UI (optional iframe)

print("Spark UI:", spark.sparkContext.uiWebUrl)

try:
    from google.colab import output  # environment-specific
    output.serve_kernel_port_as_iframe(4040, height=900)
except Exception as e:
    print("Iframe view skipped (environment not supported):", e)


## Spark History Server (optional) and event-log summaries

Event logs are enabled so runs can be reviewed after execution. Optionally start a Spark History Server and generate summary tables from event logs for inclusion in Dashboard 4 and the report.


In [ ]:
# Section: Start Spark History Server (optional)

import os, re, subprocess

event_log_dir = CONFIG["event_log_dir"]
os.makedirs(event_log_dir, exist_ok=True)

# Start history server if spark-class is available
history_cmd = ["bash", "-lc", f"$SPARK_HOME/sbin/start-history-server.sh --dir {event_log_dir} || true"]
subprocess.run(history_cmd, check=False)

print("Event logs:", event_log_dir)
print("History server (default): http://localhost:18080")

try:
    from google.colab import output  # environment-specific
    output.serve_kernel_port_as_iframe(18080, height=900)
except Exception as e:
    print("Iframe view skipped (environment not supported):", e)


In [ ]:
# ============================================================
# CELL 26 — SparkUI Boards (Jobs/Stages) from Spark event logs
# Fixes:
# - Reads Spark eventLog v2 (.zstd) safely
# - Skips appstatus files
# - Uses explicit schemas (no type inference failures)
# Outputs:
#   <gold_root>/sparkui_boards/jobs
#   <gold_root>/sparkui_boards/stages
# ============================================================

# One-time dependency for reading .zstd Spark event logs
!pip -q install zstandard

import io
import zstandard as zstd

event_log_dir = CONFIG["event_log_dir"]
boards_out = f"{paths['gold_root']}/sparkui_boards"
os.makedirs(boards_out, exist_ok=True)

print("Event log dir:", event_log_dir)

def list_eventlog_files(root_dir):
    out = []
    for root, _, files in os.walk(root_dir):
        for fn in files:
            if fn.startswith("."):
                continue
            if "appstatus" in fn.lower():
                continue
            p = os.path.join(root, fn)
            if os.path.isfile(p):
                out.append(p)
    return sorted(out)

def iter_eventlog_lines(path):
    if path.endswith(".zstd"):
        with open(path, "rb") as fh:
            dctx = zstd.ZstdDecompressor()
            with dctx.stream_reader(fh) as reader:
                text = io.TextIOWrapper(reader, encoding="utf-8", errors="ignore")
                for line in text:
                    yield line
    else:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                yield line

files = list_eventlog_files(event_log_dir)
print("Event log files found:", len(files))
if files:
    print("Example:", files[0])

def to_int(x):
    try:
        return int(x) if x is not None else None
    except Exception:
        return None

# --- Collect in fixed-structure lists (no mixed keys/types) ---
job_starts = []
job_ends = []
stage_submitted = []
stage_completed = []

for fp in files:
    print("Parsing:", fp)
    base = os.path.basename(fp)

    for line in iter_eventlog_lines(fp):
        line = line.strip()
        if not line:
            continue

        try:
            ev = json.loads(line)
        except Exception:
            continue

        if not isinstance(ev, dict):
            continue

        et = ev.get("Event")
        if et is None:
            continue

        # Jobs
        if et == "SparkListenerJobStart":
            props = ev.get("Properties")
            job_group_id = props.get("spark.jobGroup.id") if isinstance(props, dict) else None
            stage_ids = ev.get("Stage IDs")
            # Store stage_ids as JSON string to keep schema stable
            stage_ids_json = json.dumps(stage_ids) if stage_ids is not None else None

            job_starts.append({
                "job_id": to_int(ev.get("Job ID")),
                "file": base,
                "start_ms": to_int(ev.get("Submission Time")),
                "stage_ids_json": stage_ids_json,
                "job_group_id": job_group_id
            })

        elif et == "SparkListenerJobEnd":
            job_ends.append({
                "job_id": to_int(ev.get("Job ID")),
                "file": base,
                "end_ms": to_int(ev.get("Completion Time")),
                "job_result": str(ev.get("Job Result"))
            })

        # Stages
        if et == "SparkListenerStageSubmitted":
            s = ev.get("Stage Info", {})
            if isinstance(s, dict):
                stage_submitted.append({
                    "stage_id": to_int(s.get("Stage ID")),
                    "attempt_id": to_int(s.get("Stage Attempt ID")),
                    "file": base,
                    "stage_name": s.get("Stage Name"),
                    "start_ms": to_int(s.get("Submission Time")),
                    "num_tasks": to_int(s.get("Number of Tasks"))
                })

        elif et == "SparkListenerStageCompleted":
            s = ev.get("Stage Info", {})
            if isinstance(s, dict):
                stage_completed.append({
                    "stage_id": to_int(s.get("Stage ID")),
                    "attempt_id": to_int(s.get("Stage Attempt ID")),
                    "file": base,
                    "stage_name": s.get("Stage Name"),
                    "end_ms": to_int(s.get("Completion Time")),
                    "failure_reason": s.get("Failure Reason")
                })

# --- Explicit schemas (prevents CANNOT_DETERMINE_TYPE) ---
job_start_schema = T.StructType([
    T.StructField("job_id", T.IntegerType(), True),
    T.StructField("file", T.StringType(), True),
    T.StructField("start_ms", T.LongType(), True),
    T.StructField("stage_ids_json", T.StringType(), True),
    T.StructField("job_group_id", T.StringType(), True),
])

job_end_schema = T.StructType([
    T.StructField("job_id", T.IntegerType(), True),
    T.StructField("file", T.StringType(), True),
    T.StructField("end_ms", T.LongType(), True),
    T.StructField("job_result", T.StringType(), True),
])

stage_sub_schema = T.StructType([
    T.StructField("stage_id", T.IntegerType(), True),
    T.StructField("attempt_id", T.IntegerType(), True),
    T.StructField("file", T.StringType(), True),
    T.StructField("stage_name", T.StringType(), True),
    T.StructField("start_ms", T.LongType(), True),
    T.StructField("num_tasks", T.IntegerType(), True),
])

stage_comp_schema = T.StructType([
    T.StructField("stage_id", T.IntegerType(), True),
    T.StructField("attempt_id", T.IntegerType(), True),
    T.StructField("file", T.StringType(), True),
    T.StructField("stage_name", T.StringType(), True),
    T.StructField("end_ms", T.LongType(), True),
    T.StructField("failure_reason", T.StringType(), True),
])

# ----------------------------
# Build Jobs Board
# ----------------------------
if job_starts:
    starts_df = spark.createDataFrame(job_starts, schema=job_start_schema)
else:
    starts_df = None

if job_ends:
    ends_df = spark.createDataFrame(job_ends, schema=job_end_schema)
else:
    ends_df = None

if starts_df is not None:
    jobs_board = starts_df
    if ends_df is not None:
        jobs_board = (
            starts_df.join(ends_df, ["job_id", "file"], "left")
            .withColumn("duration_s", (F.col("end_ms") - F.col("start_ms")) / F.lit(1000.0))
        )
    else:
        jobs_board = jobs_board.withColumn("end_ms", F.lit(None).cast("long")) \
                               .withColumn("job_result", F.lit(None).cast("string")) \
                               .withColumn("duration_s", F.lit(None).cast("double"))

    jobs_board = jobs_board.orderBy(F.col("duration_s").desc_nulls_last())

    jobs_out = f"{boards_out}/jobs"
    jobs_board.write.mode("overwrite").parquet(jobs_out)
    print("Saved jobs board:", jobs_out)
    jobs_board.show(10, truncate=False)
else:
    print("No job events parsed.")

# ----------------------------
# Build Stages Board
# ----------------------------
sub_df = spark.createDataFrame(stage_submitted, schema=stage_sub_schema) if stage_submitted else None
comp_df = spark.createDataFrame(stage_completed, schema=stage_comp_schema) if stage_completed else None

if sub_df is not None:
    stages_board = sub_df
    if comp_df is not None:
        stages_board = (
            sub_df.join(comp_df, ["stage_id", "attempt_id", "file", "stage_name"], "left")
            .withColumn("duration_s", (F.col("end_ms") - F.col("start_ms")) / F.lit(1000.0))
        )
    else:
        stages_board = stages_board.withColumn("end_ms", F.lit(None).cast("long")) \
                                   .withColumn("failure_reason", F.lit(None).cast("string")) \
                                   .withColumn("duration_s", F.lit(None).cast("double"))

    stages_board = stages_board.orderBy(F.col("duration_s").desc_nulls_last())

    stages_out = f"{boards_out}/stages"
    stages_board.write.mode("overwrite").parquet(stages_out)
    print("Saved stages board:", stages_out)
    stages_board.show(10, truncate=False)
else:
    print("No stage events parsed.")